# 04 — Model Experiments

**Objective:** Load processed data, train each model from `src/models/`, and log results.

- Autoencoder (unsupervised, normal-only → latent features)
- Random Forest (supervised, hybrid features)
- XGBoost (supervised, hybrid features)

In [ ]:
import sys, os
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.config import get_paths, get_model_config, get_train_config, ensure_output_dirs
from src.data.preprocess import run_preprocessing
from src.features.build_features import extract_latent_features, build_hybrid_features
from src.evaluation.metrics import evaluate_model
from src.evaluation.plot_results import plot_all_for_model

ensure_output_dirs()
paths = get_paths()

## 1. Load & Preprocess Data

In [ ]:
(X_train_scaled, X_test_scaled,
 y_train, y_test,
 feature_names, scaler, encoders) = run_preprocessing()

print(f'X_train: {X_train_scaled.shape}')
print(f'X_test:  {X_test_scaled.shape}')

## 2. Train Autoencoder

In [ ]:
from src.models.autoencoder import build_autoencoder, train_autoencoder

# Select normal-only for AE training
normal_mask = (y_train == 0)
X_train_normal = X_train_scaled[normal_mask]
print(f'Normal training samples: {X_train_normal.shape[0]:,}')

input_dim = X_train_scaled.shape[1]
autoencoder, encoder = build_autoencoder(input_dim)
history_ae = train_autoencoder(autoencoder, X_train_normal)

In [ ]:
# Extract latent features and build hybrid set
latent_train, latent_test = extract_latent_features(encoder, X_train_scaled, X_test_scaled)
X_train_hybrid = build_hybrid_features(X_train_scaled, latent_train)
X_test_hybrid = build_hybrid_features(X_test_scaled, latent_test)

# Plot AE results
from src.evaluation.plot_results import plot_training_loss
plot_training_loss(history_ae, 'autoencoder')

## 3. Train Random Forest

In [ ]:
from src.models.random_forest import train_random_forest

rf_clf = train_random_forest(X_train_hybrid, y_train)

# Evaluate
rf_pred = rf_clf.predict(X_test_hybrid)
rf_proba = rf_clf.predict_proba(X_test_hybrid)[:, 1]
rf_metrics = evaluate_model(y_test, rf_pred, rf_proba, 'random_forest')

# Plots
plot_all_for_model(y_test, rf_pred, rf_proba, 'random_forest')

## 4. Train XGBoost

In [ ]:
from src.models.xgboost_model import train_xgboost

xgb_clf = train_xgboost(X_train_hybrid, y_train)

# Evaluate
xgb_pred = xgb_clf.predict(X_test_hybrid)
xgb_proba = xgb_clf.predict_proba(X_test_hybrid)[:, 1]
xgb_metrics = evaluate_model(y_test, xgb_pred, xgb_proba, 'xgboost')

# Plots
plot_all_for_model(y_test, xgb_pred, xgb_proba, 'xgboost')

## 5. Quick Comparison

In [ ]:
results = pd.DataFrame([rf_metrics, xgb_metrics])
results = results[['model_name', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']]
results = results.set_index('model_name')
print('\n=== Model Comparison ===')
display(results)